# Data Exploration & SQL Sync

This notebook verifies the data loaded from Feather files and syncs it to the SQLite database for row-level editing.

In [5]:
import sys
import os
sys.path.append(os.path.abspath('../'))

from src.data_handler import StatcastDataHandler
from src.db_manager import sync_feather_to_sql, get_db_connection
import pandas as pd

In [6]:
handler = StatcastDataHandler()

# 1. Verify Feather Data
print("--- Feather Files ---")
for i in range(2020, 2026):
    df = handler.fetch_statcast(start_year=i, end_year=i)
    print(f"Year {i}: {df.shape}")

--- Feather Files ---
Loading data from 2020 to 2020...
Year 2020: (279660, 88)
Loading data from 2021 to 2021...
Year 2021: (720638, 88)
Loading data from 2022 to 2022...
Year 2022: (720272, 88)
Loading data from 2023 to 2023...
Year 2023: (729774, 88)
Loading data from 2024 to 2024...
Year 2024: (722296, 88)
Loading data from 2025 to 2025...
Year 2025: (726773, 88)


In [3]:
# 2. Sync to SQL (One-off initialization)
# Uncomment to run the sync directly here, or run via script
# sync_feather_to_sql(range(2020, 2026))

In [4]:
# 3. Verify SQL Database
conn = get_db_connection()
try:
    count_df = pd.read_sql("SELECT count(*) as cnt FROM statcast", conn)
    print(f"\nTotal rows in SQL DB: {count_df['cnt'].iloc[0]}")
    
    # Sample query
    sample_df = pd.read_sql("SELECT game_date, pitcher_name, pitch_type, release_speed FROM statcast LIMIT 5", conn)
    print(sample_df)
except Exception as e:
    print(f"SQL Error (maybe table doesn't exist yet): {e}")
finally:
    conn.close()


Total rows in SQL DB: 3899413
             game_date pitcher_name pitch_type  release_speed
0  2020-10-27 00:00:00  Julio Urías         FF           96.7
1  2020-10-27 00:00:00  Julio Urías         SI           94.1
2  2020-10-27 00:00:00  Julio Urías         FF           94.9
3  2020-10-27 00:00:00  Julio Urías         FF           94.4
4  2020-10-27 00:00:00  Julio Urías         SV           81.4


# 4. ID Creation (SQL)

Directly alter the database to creating a unique `play_id` for each pitch.

In [21]:
# Create your own unique play_id directly in SQL
conn = get_db_connection()
cursor = conn.cursor()

# 1. Add column if not exists
try:
    cursor.execute("ALTER TABLE statcast ADD COLUMN play_id TEXT")
    print("Added 'play_id' column to statcast table.")
except Exception as e:
    print(f"'play_id' column check: {e}")

# 2. Update column (using SQLite concatenation ||)
# CAST to ensure string concatenation
print("Updating 'play_id' for all rows... (this may take a moment)")
cursor.execute("""
    UPDATE statcast 
    SET play_id = CAST(game_pk AS TEXT) || '_' || CAST(at_bat_number AS TEXT) || '_' || CAST(pitch_number AS TEXT)
    WHERE play_id IS NULL
""")

rows_updated = cursor.rowcount
conn.commit()
print(f"Updated {rows_updated} rows.")

# 3. Verify
print("\n--- Verification (First 5 Rows) ---")
sample_df = pd.read_sql("SELECT play_id, game_pk, at_bat_number, pitch_number FROM statcast LIMIT 5", conn)
print(sample_df)

conn.close()

Added 'play_id' column to statcast table.
Updating 'play_id' for all rows... (this may take a moment)
Updated 3899413 rows.

--- Verification (First 5 Rows) ---
       play_id  game_pk  at_bat_number  pitch_number
0  635886_65_3   635886             65             3
1  635886_65_2   635886             65             2
2  635886_65_1   635886             65             1
3  635886_64_6   635886             64             6
4  635886_64_5   635886             64             5
